# Exploratory Data Analysis (EDA) — Peta Digital UMKM Kuliner Surabaya Timur
### Analisis Multimodal: Tabular, Teks Ulasan, Citra Foto, dan Dokumen Regulasi

**Identitas Mahasiswa & Kelompok:**
- **NIM:** 09040624091
- **Nama:** Muhammad Iqbal Rizanta
- **Kelompok:** 3
- **Mata Kuliah:** Big Data Analytics
- **Program Studi:** Sistem Informasi, Fakultas Sains dan Teknologi
- **Institusi:** Universitas Islam Negeri Sunan Ampel Surabaya
- **Dosen Pengampu:** Subhan Nooriansyah, M.Kom.

---
## 1. Pendahuluan & Ringkasan Dataset
Notebook ini memuat analisis eksplorasi data (EDA) komprehensif pada **4 modalitas data**:
1. **Data Tabular:** Metadata profil 147 UMKM kuliner di 6 kecamatan Surabaya Timur + stan survei.
2. **Data Teks:** 730 ulasan pengunjung dari Google Places API setelah disanitasi & dinormalisasi.
3. **Data Citra/Foto:** 336 foto (292 foto Google Maps dan 44 foto survei lapangan SWK Convention Hall & Deles) setelah proses anonimisasi privasi (face blurring).
4. **Data Dokumen:** 3 dokumen perizinan & proposal penelitian dari Dinas Koperasi & UINSA.


In [ ]:
# 1. Import Pustaka Analisis & Visualisasi
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Konfigurasi Tampilan Visualisasi
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120
print('Pustaka berhasil diimpor!')


---
## 2. Pemuatan Data dari Zona Clean (Format Parquet)
Data yang dianalisis berasal langsung dari zona `data/clean/` yang telah divalidasi, dideduplikasi, dan dianonimkan.


In [ ]:
# 2. Load Dataset Bersih Parquet
clean_dir = Path('../data/clean') if Path('../data/clean').exists() else Path('data/clean')

df_umkm = pd.read_parquet(clean_dir / 'umkm.parquet')
df_ulasan = pd.read_parquet(clean_dir / 'ulasan.parquet')
df_foto = pd.read_parquet(clean_dir / 'foto.parquet')
df_dokumen = pd.read_parquet(clean_dir / 'dokumen.parquet')

print(f'Total UMKM Terdaftar   : {len(df_umkm)}')
print(f'Total Ulasan Bersih    : {len(df_ulasan)}')
print(f'Total Foto Terverifikasi: {len(df_foto)}')
print(f'Total Dokumen Perizinan: {len(df_dokumen)}')


---
## 3. EDA Modalitas 1 — Data Tabular UMKM Kuliner
Menganalisis sebaran jumlah UMKM per kecamatan di Surabaya Timur, distribusi rating publik, dan proporsi Sentra Wisata Kuliner (SWK).


In [ ]:
# 3.1 Ringkasan Struktur Data UMKM
print('Struktur Kolom df_umkm:')
df_umkm.info()
display(df_umkm.head(5))


In [ ]:
# 3.2 Visualisasi Sebaran UMKM per Kecamatan
plt.figure(figsize=(10, 5))
kec_counts = df_umkm['kecamatan'].value_counts()
sns.barplot(x=kec_counts.values, y=kec_counts.index, palette='viridis')
plt.title('Distribusi Titik UMKM Kuliner per Kecamatan di Surabaya Timur', fontsize=14, fontweight='bold')
plt.xlabel('Jumlah UMKM', fontsize=12)
plt.ylabel('Kecamatan', fontsize=12)
for i, v in enumerate(kec_counts.values):
    plt.text(v + 0.5, i, str(v), color='black', va='center', fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# 3.3 Distribusi Rating Google Maps
plt.figure(figsize=(8, 4))
sns.histplot(df_umkm['rating'].dropna(), bins=15, kde=True, color='teal')
plt.title('Distribusi Rating Google Maps UMKM Kuliner Surabaya Timur', fontsize=13, fontweight='bold')
plt.xlabel('Rating (1.0 - 5.0)', fontsize=11)
plt.ylabel('Frekuensi', fontsize=11)
plt.tight_layout()
plt.show()

print('Statistik Deskriptif Rating:')
print(df_umkm['rating'].describe())


---
## 4. EDA Modalitas 2 — Data Teks Ulasan Pelanggan
Menganalisis panjang ulasan, sentimen awal, dan frekuensi penyebutan kata kunci pembayaran digital (QRIS, POS, EDC, Merchant Online).


In [ ]:
# 4.1 Frekuensi Sinyal Digital dalam Ulasan
sinyal_cols = ['mention_qris', 'mention_pos', 'mention_edc', 'mention_online']
sinyal_sum = df_ulasan[sinyal_cols].sum()

plt.figure(figsize=(8, 4))
labels = ['QRIS / Barcode', 'POS / Kasir', 'EDC / Debit', 'Merchant Online (Ojol)']
sns.barplot(x=labels, y=sinyal_sum.values, palette='magma')
plt.title('Frekuensi Penyebutan Indikator Digital dalam Ulasan Pelanggan', fontsize=13, fontweight='bold')
plt.ylabel('Jumlah Kemunculan dalam Ulasan', fontsize=11)
for i, v in enumerate(sinyal_sum.values):
    plt.text(i, v + 1, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# 4.2 Analisis Panjang Kata Ulasan
plt.figure(figsize=(8, 4))
sns.boxplot(x=df_ulasan['kecamatan'], y=df_ulasan['word_count'], palette='Set2')
plt.title('Panjang Kata Ulasan Pelanggan per Kecamatan', fontsize=13, fontweight='bold')
plt.xlabel('Kecamatan', fontsize=11)
plt.ylabel('Jumlah Kata', fontsize=11)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


---
## 5. EDA Modalitas 3 — Citra Foto & Ground Truth Survei SWK
Menganalisis distribusi foto online vs survei lapangan SWK (Convention Hall & Deles), serta sebaran kelas kematangan digital ground truth.


In [ ]:
# 5.1 Distribusi Kelas Ground Truth Survei Lapangan SWK
df_survei_foto = df_foto[df_foto['sumber'] == 'survei_lapangan_swk']
class_counts = df_survei_foto['label_ground_truth'].value_counts()

plt.figure(figsize=(7, 5))
colors = ['#ff9999', '#66b3ff', '#99ff99']
plt.pie(class_counts.values, labels=class_counts.index, autopct='%1.1f%%', colors=colors, startangle=140, explode=(0.05, 0.05, 0.05))
plt.title('Distribusi Kelas Ground Truth Survei SWK\n(Convention Hall & Deles)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Rincian Foto Survei Lapangan:')
for k, v in class_counts.items():
    print(f'  - Kelas {k}: {v} foto')


In [ ]:
# 5.2 Status Anonimisasi Privasi (Blur Wajah)
total_blurred = df_foto['face_blurred_count'].sum()
photos_with_faces = len(df_foto[df_foto['face_blurred_count'] > 0])

print(f'Total wajah pengunjung/kasir yang terdeteksi & diburamkan: {total_blurred} wajah')
print(f'Jumlah foto yang memuat wajah dan telah disanitasi: {photos_with_faces} foto')


---
## 6. EDA Modalitas 4 — Dokumen Regulasi & Perizinan
Mengevaluasi dokumen perizinan penelitian dari Dinas Koperasi Usaha Kecil Menengah dan Perdagangan Kota Surabaya.


In [ ]:
# 6.1 Ringkasan Dokumen
display(df_dokumen[['dok_id', 'file_name', 'pages', 'file_size_kb', 'institusi']])

print('\nContoh Ringkasan Teks Dokumen:')
print(df_dokumen.loc[0, 'ringkasan_teks'][:350] + '...')


---
## 7. Kesimpulan & Temuan Kunci EDA
1. **Representasi Spasial:** 6 kecamatan di Surabaya Timur terpetakan dengan baik (total 147 UMKM reguler + stan sentra kuliner).
2. **Pola Pembayaran Digital:** QRIS dan Merchant Online (GoFood/GrabFood) merupakan sinyal digital yang paling sering muncul di ulasan pelanggan.
3. **Kualitas Ground Truth SWK:** Data survei offline dari SWK Convention Hall dan SWK Deles menyediakan 44 foto teranotasi dengan sebaran kelas yang seimbang (9 Konvensional, 16 Berkembang, 19 Maju).
4. **Kepatuhan Privasi:** 100% ulasan teks telah disanitasi dari nomor telepon/identitas pengulas, dan seluruh wajah pada foto telah diburamkan menggunakan OpenCV Gaussian Blur sebelum masuk ke tahap modeling.
